# Solutions Notebook — Time Series Diagnostics

Full worked solutions for `03_Skeleton_Practice_Notebook.ipynb` (same task numbering). Try the skeleton first — use this only to check your work or get unstuck.


## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.seasonal import seasonal_decompose
from scipy.signal import correlate

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 4)


## Task A1

In [ ]:
n = 600
rng = np.random.default_rng(7)

phi = 0.6
eps = rng.normal(0, 1, n)
ar1 = np.zeros(n)
for t in range(1, n):
    ar1[t] = phi * ar1[t-1] + eps[t]

regime = np.concatenate([rng.normal(0, 1, n//2), rng.normal(0, 4, n//2)])

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(ar1); ax[0].set_title("Stationary AR(1)")
ax[1].plot(regime); ax[1].set_title("Variance regime shift")
plt.tight_layout(); plt.show()

print("AR(1):    first half", (ar1[:n//2].mean(), ar1[:n//2].std()),
      " | second half", (ar1[n//2:].mean(), ar1[n//2:].std()))
print("Regime:   first half", (regime[:n//2].mean(), regime[:n//2].std()),
      " | second half", (regime[n//2:].mean(), regime[n//2:].std()))


**Interpretation:** the AR(1) series' mean/std are similar between halves (ergodic-like — a single stretch is representative). The regime-shift series has similar means but very different standard deviations between halves — a statistic from the first half would badly mislead you about the second.

## Task A2

In [ ]:
rng = np.random.default_rng(11)
n = 300
step = np.concatenate([np.zeros(n//2), np.full(n//2, 5.0)])
mean_shift_series = step + rng.normal(0, 1, n)

plt.plot(mean_shift_series); plt.title("Mean-shift series"); plt.show()

thirds = np.array_split(mean_shift_series, 3)
for i, part in enumerate(thirds, 1):
    print(f"Third {i} mean: {part.mean():.3f}")


**Conclusion:** the three thirds have very different means (~0, transitional, ~5), so the *full-sample* mean is a poor summary and a poor forecast of "what comes next" — this is a non-ergodic-flavored series where recent history, not the whole history, is what matters.

## Task B1

In [ ]:
random_white_noise = np.random.normal(loc=0, scale=1, size=1000)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(random_white_noise); ax[0].axhline(0, color='r'); ax[0].set_title('Raw Data')
plot_acf(random_white_noise, ax=ax[1])
plt.tight_layout(); plt.show()

acorr_ljungbox(random_white_noise, lags=[10, 30, 50], return_df=True)


**Interpretation:** all p-values are well above 0.05, so we fail to reject $H_0$ (no autocorrelation) at every tested horizon — consistent with white noise, as expected since we generated it that way.

## Task B2

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(ar1); ax[0].set_title('AR(1) series')
plot_acf(ar1, ax=ax[1])
plt.tight_layout(); plt.show()

acorr_ljungbox(ar1, lags=[10, 30, 50], return_df=True)


**Interpretation:** the p-values are essentially 0, so we reject $H_0$ — there IS significant autocorrelation. That's expected: `ar1` was built with `x_t = 0.6*x_{t-1} + eps_t`, so by construction each value is correlated with its own recent past.

## Task B3

In [ ]:
for seed in [1, 2, 3, 4, 5]:
    rng_s = np.random.default_rng(seed)
    for size in [30, 200, 2000]:
        wn = rng_s.normal(0, 1, size)
        p = acorr_ljungbox(wn, lags=[10], return_df=True)['lb_pvalue'].iloc[0]
        flag = "  <-- false alarm (p<0.05)" if p < 0.05 else ""
        print(f"seed={seed} size={size:5d}  p={p:.4f}{flag}")


**Interpretation:** occasionally (roughly 1 in 20 runs, matching the 5% significance level) a p-value below 0.05 shows up even though the data is genuinely white noise — a **Type I error**. This is why a single hypothesis test at a single lag/seed shouldn't be treated as absolute proof; combine it with visual inspection (ACF plot) and, if possible, multiple lags/samples.

## Task C1

In [ ]:
df = sm.datasets.macrodata.load().data
df['realinv'] = round(df['realinv'].astype('float32'), 2)
df['realdpi'] = round(df['realdpi'].astype('float32'), 2)
df_mod = df[['realinv', 'realdpi']]
df_mod.head()


## Task C2

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(16, 8))
ax[0, 2].plot(df_mod['realinv']); ax[0, 2].set_title('Original Data (realinv)')
plot_acf(df_mod['realinv'], alpha=0.05, lags=50, ax=ax[0, 0]); ax[0, 0].set_title('Original ACF')
plot_pacf(df_mod['realinv'], alpha=0.05, lags=50, ax=ax[0, 1]); ax[0, 1].set_title('Original PACF')

diffed = np.diff(df_mod['realinv'], n=1)
ax[1, 2].plot(diffed); ax[1, 2].set_title('First-Differenced Data')
plot_acf(diffed, alpha=0.05, lags=50, ax=ax[1, 0]); ax[1, 0].set_title('Differenced ACF')
plot_pacf(diffed, alpha=0.05, lags=50, ax=ax[1, 1]); ax[1, 1].set_title('Differenced PACF')
plt.tight_layout(); plt.show()


## Task C3

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(16, 8))
ax[0, 2].plot(df_mod['realdpi']); ax[0, 2].set_title('Original Data (realdpi)')
plot_acf(df_mod['realdpi'], alpha=0.05, lags=50, ax=ax[0, 0]); ax[0, 0].set_title('Original ACF')
plot_pacf(df_mod['realdpi'], alpha=0.05, lags=50, ax=ax[0, 1]); ax[0, 1].set_title('Original PACF')

diffed_dpi = np.diff(df_mod['realdpi'], n=1)
ax[1, 2].plot(diffed_dpi); ax[1, 2].set_title('First-Differenced Data')
plot_acf(diffed_dpi, alpha=0.05, lags=50, ax=ax[1, 0]); ax[1, 0].set_title('Differenced ACF')
plot_pacf(diffed_dpi, alpha=0.05, lags=50, ax=ax[1, 1]); ax[1, 1].set_title('Differenced PACF')
plt.tight_layout(); plt.show()


**Comparison:** both series show the same qualitative pattern — a slowly-decaying original ACF (trend-dominated) that becomes short-range after one difference. The differenced ACF/PACF shapes are broadly similar in spirit (a small number of significant early lags), suggesting comparably low-order AR/MA candidates for each, though the exact significant lags differ slightly.

## Task C4

In [ ]:
diffed_twice = np.diff(diffed, n=1)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
plot_acf(diffed_twice, alpha=0.05, lags=50, ax=ax[0]); ax[0].set_title('Twice-Differenced ACF')
plot_pacf(diffed_twice, alpha=0.05, lags=50, ax=ax[1]); ax[1].set_title('Twice-Differenced PACF')
plt.tight_layout(); plt.show()


**Interpretation:** the twice-differenced ACF typically shows a pronounced negative spike at lag 1 that wasn't there (or was much smaller) after a single difference — a textbook sign of over-differencing. This tells us one difference was already enough; a second difference introduces artificial negative correlation rather than removing more real structure.

## Task D1

In [ ]:
def plot_ccf(data_a, data_b, lag_lookback, percentile=95):
    n = len(data_a)
    ccf = correlate(data_a - np.mean(data_a), data_b - np.mean(data_b), method='direct') / (
        np.std(data_a) * np.std(data_b) * n
    )
    _min = (len(ccf) - 1) // 2 - lag_lookback
    _max = (len(ccf) - 1) // 2 + (lag_lookback - 1)
    zscore_vals = {90: 1.645, 95: 1.96, 99: 2.576}
    z = zscore_vals[percentile] / np.sqrt(n)

    plt.figure(figsize=(13, 4))
    markers, stems, baseline = plt.stem(
        np.arange(-lag_lookback, lag_lookback - 1), ccf[_min:_max], markerfmt='o'
    )
    plt.setp(baseline, color='r', linewidth=1)
    plt.axhline(y=z, color='b', ls='--')
    plt.axhline(y=-z, color='b', ls='--')
    plt.axvline(x=0, color='black', ls='-')
    plt.title('Cross-Correlation'); plt.xlabel('Lag'); plt.ylabel('Correlation')
    plt.show()
    return ccf


## Task D2

In [ ]:
df_diff = pd.DataFrame()
df_diff['realinv'] = np.diff(df_mod['realinv'], n=1)
df_diff['realdpi'] = np.diff(df_mod['realdpi'], n=1)

_ = plot_ccf(data_a=df_diff['realdpi'], data_b=df_diff['realinv'], lag_lookback=50, percentile=95)


**Interpretation:** the peak correlation sits at lag 0 (~0.2 magnitude), so the two series are contemporaneously related rather than one leading the other, and the correlation is fairly weak in practical terms.

## Task D3

In [ ]:
rng = np.random.default_rng(3)
weeks = 300
ad_spend = rng.normal(0, 1, weeks)
sales_signal = np.roll(ad_spend, 2) * 0.8
sales = sales_signal + rng.normal(0, 0.5, weeks)
sales[:2] = rng.normal(0, 0.5, 2)  # fix wrap-around

_ = plot_ccf(data_a=ad_spend, data_b=sales, lag_lookback=15, percentile=95)

# Align and confirm the peak moves to lag 0
ad_spend_series = pd.Series(ad_spend)
aligned_ad_spend = ad_spend_series.shift(2).iloc[2:].values
aligned_sales = sales[2:]
_ = plot_ccf(data_a=aligned_ad_spend, data_b=aligned_sales, lag_lookback=15, percentile=95)


**Interpretation:** the first CCF peaks at lag 2, matching how `sales` was constructed to respond to `ad_spend` from 2 periods earlier. After shifting `ad_spend` forward by 2 with `.shift(2)`, the peak moves to lag 0 — this is the version of the two series you'd actually feed into a same-time-step regression model.

## Task E1

In [ ]:
data = pd.read_csv('airline-passengers.csv', header=0, index_col=0)
data.index = pd.to_datetime(data.index, format='%Y-%m')
plt.plot(data); plt.title('Monthly Airline Passengers, 1949-1960'); plt.show()


## Task E2

In [ ]:
season_trend = seasonal_decompose(data, model='additive')
season_trend.plot()
plt.show()

fig, ax = plt.subplots(figsize=(16, 6))
sns.boxplot(x=data.index.year, y=data['Passengers'], ax=ax, color="cornflowerblue")
ax.set(xlabel='Year', ylabel='Number of Passengers')
plt.show()


**Interpretation:** the decomposition shows a clearly rising trend and a repeating annual seasonal pattern (summer peaks). The year-by-year boxplots show both rising level (medians climb every year) and rising spread (taller boxes/whiskers in later years) — direct visual violations of the constant-mean and constant-variance stationarity conditions.

## Task E3

In [ ]:
plot_acf(data, lags=20, alpha=0.05); plt.title("ACF on raw (trending) data"); plt.show()
print(acorr_ljungbox(data, lags=[50], return_df=True))

def stationarity_report(series, name="series"):
    series = pd.Series(series).dropna()
    adf_stat, adf_p, *_ = adfuller(series, autolag='AIC')
    kpss_stat, kpss_p, *_ = kpss(series, regression='c', nlags='auto')
    print(f"--- {name} ---")
    print(f"ADF:  stat={adf_stat:.3f}, p-value={adf_p:.4f}  -> {'stationary' if adf_p < 0.05 else 'NON-stationary'}")
    print(f"KPSS: stat={kpss_stat:.3f}, p-value={kpss_p:.4f}  -> {'NON-stationary' if kpss_p < 0.05 else 'stationary'}")
    print()

stationarity_report(data['Passengers'], "raw")
stationarity_report(np.diff(data['Passengers'], n=1), "1st difference")


**Interpretation:** the raw series' ACF decays very slowly (dominated by trend) and the Ljung-Box test rejects the no-autocorrelation null overwhelmingly (p≈0). ADF typically fails to reject non-stationarity on the raw series, while a single first difference is often enough to flip the ADF conclusion — though, as Task E4 shows, seasonality can still linger.

## Task E4

In [ ]:
log_passengers = np.log(data['Passengers'])
seasonal_diff_of_log = log_passengers.diff(12).dropna()
first_diff_of_seasonal_diff = seasonal_diff_of_log.diff(1).dropna()

fig, ax = plt.subplots(3, 1, figsize=(10, 9))
ax[0].plot(log_passengers); ax[0].set_title('log(Passengers)')
ax[1].plot(seasonal_diff_of_log); ax[1].set_title('Seasonal diff (lag 12) of log(Passengers)')
ax[2].plot(first_diff_of_seasonal_diff); ax[2].set_title('+ first diff')
plt.tight_layout(); plt.show()

stationarity_report(log_passengers, "log(Passengers)")
stationarity_report(seasonal_diff_of_log, "log + seasonal diff(12)")
stationarity_report(first_diff_of_seasonal_diff, "log + seasonal diff(12) + 1st diff")


**Interpretation:** the log transform equalizes the growing seasonal amplitude (turns the multiplicative seasonal swing into a roughly constant-amplitude additive one). The seasonal difference (lag 12) removes the repeating annual cycle; a further first difference then removes the residual slow drift in level. The combination typically passes both ADF and KPSS as stationary, whereas a plain first difference of the raw series alone often still shows leftover seasonal structure in its ACF.

## Task F1 & F2

In [ ]:
rng = np.random.default_rng(2024)
n = 400
t = np.arange(n)

series1 = rng.normal(0, 1, n)                      # white noise
series2 = np.cumsum(rng.normal(0, 1, n))           # random walk

ar2 = np.zeros(n)
e = rng.normal(0, 1, n)
for i in range(2, n):
    ar2[i] = 0.5 * ar2[i-1] - 0.3 * ar2[i-2] + e[i]
seasonal_component = 3 * np.sin(2 * np.pi * t / 12)
series3 = ar2 + seasonal_component            # stationary AR(2) + seasonal

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(series1); ax[0].set_title('Series 1')
ax[1].plot(series2); ax[1].set_title('Series 2')
ax[2].plot(series3); ax[2].set_title('Series 3')
plt.tight_layout(); plt.show()


In [ ]:
def full_diagnostic(series, name):
    print("="*60, name, "="*60)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    plot_acf(series, ax=ax[0], lags=36); ax[0].set_title(f'{name} ACF')
    plot_pacf(series, ax=ax[1], lags=36); ax[1].set_title(f'{name} PACF')
    plt.tight_layout(); plt.show()
    print(acorr_ljungbox(series, lags=[10, 20], return_df=True))
    stationarity_report(series, name)

full_diagnostic(series1, "Series 1 (white noise)")
full_diagnostic(series2, "Series 2 (random walk)")
full_diagnostic(series3, "Series 3 (AR(2)+seasonal)")


In [ ]:
# Series 2 needs differencing:
series2_diff = np.diff(series2, n=1)
full_diagnostic(series2_diff, "Series 2, first-differenced")

# Series 3 needs seasonal differencing (period 12):
series3_seasonal_diff = pd.Series(series3).diff(12).dropna()
full_diagnostic(series3_seasonal_diff, "Series 3, seasonal-differenced (12)")


**Verdicts:**

- **Series 1 (white noise):** Ljung-Box p-values are large at every lag; ADF rejects the unit-root null strongly; KPSS fails to reject stationarity. No transform needed. No AR/MA structure — best forecast is the mean.
- **Series 2 (random walk):** ACF decays very slowly (near 1 at low lags); ADF fails to reject non-stationarity; KPSS rejects stationarity. After a first difference, it reduces back to white noise (matching how it was built as a cumulative sum of noise) — an ARIMA(0,1,0) / "random walk" model, not an AR/MA model on the levels.
- **Series 3 (AR(2) + seasonal):** raw ACF/PACF are contaminated by the periodic component (repeating spikes near multiples of 12); ADF/KPSS may or may not agree depending on the current run's noise draw. After a seasonal difference (lag 12), the PACF shows the AR(2) direct structure cutting off after lag 2, consistent with how it was generated (AR order 2, with any remaining pattern attributable to noise).
